# Mat & Comp – Python 12: Proyecto integrador
## Picas y Fijas: un juego completo que no se cae

**Curso:** Python desde cero (Sección 1) · **Duración sugerida:** 1 o 2 sesiones de 50 minutos · **Requisito:** Python 11

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Leer un mensaje de error y reconocer los tipos más comunes (`ValueError`, `ZeroDivisionError`, `IndexError`, `KeyError`…).
2. **Manejar errores** con `try` / `except` para que un programa no se caiga cuando alguien escribe algo inesperado.
3. Lanzar tus propios errores con `raise` y probar tus funciones con `assert`.
4. Dividir un problema grande en **funciones pequeñas** y juntarlas en un juego completo.
5. Programar a la computadora para que juegue sola, y analizar sus partidas con un diccionario y una gráfica.

## 🔍 ¿Puedes adivinarlo en 7 intentos?

**Picas y Fijas** es un juego de lápiz y papel muy popular en Colombia. Un jugador piensa un número de **4 cifras distintas**, por ejemplo `4271`. El otro intenta adivinarlo, y después de cada intento recibe una pista:

- una **fija** es una cifra correcta **en el lugar correcto**;
- una **pica** es una cifra correcta **en el lugar equivocado**.

Si el secreto es `4271` y alguien dice `1234`, el 2 está en su lugar (1 fija), y el 1 y el 4 están en el número pero en otro lugar (2 picas).

✏️ *Hay 5040 números posibles. ¿Crees que se puede adivinar cualquiera en 7 intentos o menos usando solo las pistas? Al final de la lección la computadora jugará cientos de partidas y lo averiguaremos.*

Hoy vamos a programar este juego usando **todo** lo que aprendiste en la sección: variables, textos, condicionales, ciclos, listas, funciones, diccionarios, gráficas y numpy. Pero primero necesitamos una herramienta nueva: el **manejo de errores**.

## 1. Cuando el programa se cae

Ya has visto muchos errores. Ejecuta esta celda: el jugador escribió la palabra "tres" en lugar del número.

In [ ]:
texto = "tres"
n = int(texto)
print("El doble es", 2 * n)

Python se detiene y muestra un mensaje (se llama *traceback*). Lo más importante está en la **última línea**: el **tipo** de error (`ValueError`) y una explicación (`invalid literal for int()`, «texto inválido para convertir a entero»).

Estos son los tipos de error que más vas a encontrar:

| Error | Cuándo aparece | Ejemplo |
|---|---|---|
| `ValueError` | El valor no sirve, aunque el tipo sea el correcto | `int("tres")` |
| `ZeroDivisionError` | Dividir entre cero | `5 / 0` |
| `IndexError` | Pedir una posición que no existe en una lista | `[1, 2, 3][10]` |
| `KeyError` | Pedir una llave que no está en un diccionario | `{"a": 1}["b"]` |
| `TypeError` | Operar con tipos que no combinan | `"3" + 4` |
| `NameError` | Usar una variable que no existe (¡casi siempre un error de escritura!) | `pirnt("hola")` |

Un programa para personas **no se puede caer** cada vez que alguien se equivoca al escribir. Para eso existe `try`.

## 2. `try` y `except`: intentar y tener un plan B

```python
try:
    # código que podría fallar
except TipoDeError:
    # qué hacer si falla con ese tipo de error
```

Python **intenta** ejecutar el bloque `try`. Si todo sale bien, se salta el `except`. Si aparece un error del tipo indicado, abandona el `try` en esa línea y ejecuta el bloque `except`. En los dos casos, el programa **sigue vivo**.

In [ ]:
texto = "tres"

try:
    n = int(texto)
    print("El doble es", 2 * n)      # esta línea no se ejecuta si int() falla
except ValueError:
    print("Eso no es un número entero:", texto)

print("El programa sigue vivo 💪")

Probemos con varias entradas distintas. Antes de ejecutar, ✏️ *predice qué pasará con cada una*.

In [ ]:
entradas = ["7", "siete", "3.5", "-2", " 12 ", ""]

for texto in entradas:
    try:
        n = int(texto)
        print(f"{texto!r:8} → el doble es {2 * n}")
    except ValueError:
        print(f"{texto!r:8} → no es un entero")

💡 `{texto!r}` muestra el texto **con comillas**, para ver los espacios o el texto vacío. Fíjate que `int(" 12 ")` sí funciona: `int` ignora los espacios de los extremos.

### Varios `except`

Una función puede fallar de varias maneras. Podemos dar un plan B distinto a cada tipo de error:

In [ ]:
def dividir(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("  No se puede dividir entre cero.")
    except TypeError:
        print("  Solo se pueden dividir números.")
    return None          # llegamos aquí solo si hubo un error

print(dividir(10, 4))
print(dividir(10, 0))
print(dividir("10", 2))

## 3. Pedir un dato hasta que sea válido

Este es el uso más común de `try` en juegos: preguntar **una y otra vez** hasta que el jugador escriba algo que sirva. Combinamos un ciclo `while True` con `try`:

- si `int()` funciona, el `return` entrega el número y termina la función (y el ciclo);
- si falla, el `except` muestra un aviso y el ciclo vuelve a preguntar.

In [ ]:
def leer_entero(mensaje):
    while True:
        texto = input(mensaje)
        try:
            return int(texto)
        except ValueError:
            print("  Escribe un número entero, por ejemplo 15.")

edad = leer_entero("¿Cuántos años tienes? ")
print("En 10 años tendrás", edad + 10)

✏️ *Ejecuta la celda y escribe primero algo que no sea un número, como `quince` o `15 años`. El programa no se cae: vuelve a preguntar.*

## 4. `raise`: lanzar tus propios errores

A veces un dato tiene el tipo correcto pero **rompe las reglas del juego**. En Picas y Fijas, `1123` es un número, pero tiene cifras repetidas. Con `raise` podemos lanzar un error con **nuestro propio mensaje**:

```python
raise ValueError("explicación del problema")
```

Esta función revisa un intento y lanza un `ValueError` si algo está mal. Si todo está bien, no hace nada.

In [ ]:
def validar_intento(texto, n=4):
    if len(texto) != n:
        raise ValueError(f"Debe tener exactamente {n} cifras.")
    if not texto.isdigit():
        raise ValueError("Solo puede tener cifras del 0 al 9.")
    if len(set(texto)) != n:
        raise ValueError("Las cifras no se pueden repetir.")

💡 `texto.isdigit()` es `True` si todos los caracteres son cifras. `set(texto)` es el **conjunto** de caracteres sin repeticiones: `set("1123")` es `{'1', '2', '3'}`, que tiene 3 elementos y no 4.

Para atrapar el error **y leer su mensaje** se escribe `except ValueError as e`. La variable `e` guarda el error, y `print(e)` muestra el mensaje:

In [ ]:
for texto in ["1234", "123", "12a4", "1123", "0987"]:
    try:
        validar_intento(texto)
        print(f"{texto}: ✅ válido")
    except ValueError as e:
        print(f"{texto}: ❌ {e}")

## 5. Diseñar el juego: dividir en funciones

Un programa grande se construye como un edificio: por partes, y probando cada parte antes de unirlas. Nuestro juego necesita cuatro funciones:

| Función | Qué hace | Lo que usa |
|---|---|---|
| `generar_secreto()` | Elige un número al azar de 4 cifras distintas | `random`, listas, `join` |
| `validar_intento(texto)` | Revisa que el intento cumpla las reglas | `if`, `raise` *(¡ya la tenemos!)* |
| `contar_picas_fijas(secreto, intento)` | Calcula la pista | `for`, `in`, tuplas |
| `jugar()` | Junta todo: pregunta, valida, da pistas, decide si ganaste | `while`, `input`, `try` |

### El secreto

`random.sample(lista, 4)` elige 4 elementos **distintos** de una lista, en orden al azar. Luego `"".join(...)` los pega en un solo texto.

In [ ]:
import random

def generar_secreto(n=4):
    cifras = random.sample("0123456789", n)    # n cifras distintas, en desorden
    return "".join(cifras)

for _ in range(5):
    print(generar_secreto())

Guardamos el número como **texto** y no como entero, por dos razones: así podemos mirar cada cifra con `secreto[i]`, y así un secreto como `0587` no pierde su cero inicial.

### Las pistas

In [ ]:
def contar_picas_fijas(secreto, intento):
    fijas = 0
    comunes = 0                       # cifras del intento que están en el secreto
    for i in range(len(secreto)):
        if intento[i] == secreto[i]:
            fijas += 1
        if intento[i] in secreto:
            comunes += 1
    picas = comunes - fijas           # comunes que NO están en su lugar
    return picas, fijas               # una tupla con dos números

print(contar_picas_fijas("4271", "1234"))

## 6. Probar con `assert`

¿Cómo sabemos que `contar_picas_fijas` funciona en **todos** los casos? Escribimos pruebas. `assert condición` no hace nada si la condición es verdadera, y lanza un `AssertionError` si es falsa. Es una forma de decir: *«esto tiene que cumplirse; si no, avísame»*.

In [ ]:
assert contar_picas_fijas("4271", "1234") == (2, 1)
assert contar_picas_fijas("4271", "4271") == (0, 4)   # ganó
assert contar_picas_fijas("4271", "1742") == (4, 0)   # todas en otro lugar
assert contar_picas_fijas("4271", "5689") == (0, 0)   # ninguna
assert contar_picas_fijas("0587", "0578") == (2, 2)
print("✅ Todas las pruebas pasaron")

Si algún día cambias la función y te equivocas, esta celda te avisará de inmediato. Los programadores profesionales escriben miles de pruebas así.

✏️ *Escribe una prueba que falle a propósito (por ejemplo, `assert contar_picas_fijas("1234", "1234") == (4, 0)`) y mira el mensaje.*

In [ ]:
# Tu prueba aquí

## 7. 🎮 ¡A jugar!

Ahora juntamos todo. Lee la función con calma: cada parte usa algo que ya aprendiste.

- `input(...).strip()` lee el intento y le quita los espacios de los extremos.
- Si el jugador escribe `salir`, `return` termina el juego.
- `try` / `except` atrapa los intentos inválidos; `continue` vuelve a preguntar **sin gastar un intento**.
- La función **devuelve** el número de intentos si ganas, o `None` si pierdes. Eso nos servirá para llevar un marcador.

In [ ]:
def jugar(max_intentos=10):
    secreto = generar_secreto()
    print("Pensé un número de 4 cifras distintas. Escribe 'salir' para rendirte.")
    intentos = 0
    while intentos < max_intentos:
        texto = input(f"Intento {intentos + 1}: ").strip()
        if texto == "salir":
            print("Te rendiste. El número era", secreto)
            return None
        try:
            validar_intento(texto)
        except ValueError as e:
            print("   ⚠️", e, "(no cuenta como intento)")
            continue
        intentos += 1
        picas, fijas = contar_picas_fijas(secreto, texto)
        if fijas == 4:
            print(f"   🎉 ¡Ganaste en {intentos} intentos!")
            return intentos
        print(f"   picas: {picas}   fijas: {fijas}")
    print("Se acabaron los intentos. El número era", secreto)
    return None

resultado = jugar()

✏️ *Juega al menos una partida completa. Prueba también escribir algo inválido, como `12a4` o `1123`. ¿Cuántos intentos necesitaste?*

💡 **Estrategia:** después de cada pista, piensa qué números **siguen siendo posibles**. Eso es exactamente lo que haremos que haga la computadora.

## 8. La computadora juega sola

Una estrategia sencilla y poderosa:

1. Hacer la lista de **todos** los secretos posibles.
2. Intentar el primero de la lista.
3. Con la pista recibida, **tachar** todos los números que habrían dado una pista distinta. El secreto nunca se tacha, porque el secreto sí da esa pista.
4. Repetir hasta acertar.

Primero, la lista de candidatos. `str(n).zfill(4)` convierte el número en texto y le agrega ceros a la izquierda hasta tener 4 caracteres (`str(587).zfill(4)` es `"0587"`).

In [ ]:
candidatos = []
for n in range(10000):
    texto = str(n).zfill(4)
    if len(set(texto)) == 4:          # 4 cifras distintas
        candidatos.append(texto)

print(len(candidatos), "secretos posibles")
print("Los primeros:", candidatos[:6])
print("Los últimos:", candidatos[-3:])

¿Por qué 5040? Para la primera cifra hay 10 opciones, para la segunda quedan 9, luego 8 y luego 7: $10 \cdot 9 \cdot 8 \cdot 7 = 5040$.

Ahora la estrategia. Veamos una partida paso a paso, con el secreto `4271`:

In [ ]:
def computadora_juega(secreto, mostrar=False):
    posibles = candidatos[:]                 # una copia de la lista completa
    intentos = 0
    while True:
        intento = posibles[0]
        intentos += 1
        pista = contar_picas_fijas(secreto, intento)
        if mostrar:
            print(f"Intento {intentos}: {intento} → picas: {pista[0]}, fijas: {pista[1]}", end="")
        if pista[1] == 4:
            if mostrar:
                print("  🎉")
            return intentos
        # Tachar: quedan solo los números que habrían dado la misma pista
        posibles = [c for c in posibles if contar_picas_fijas(c, intento) == pista]
        if mostrar:
            print(f"   (quedan {len(posibles)} posibles)")

n = computadora_juega("4271", mostrar=True)

¡Mira cómo se derrumba la lista de posibles con cada pista! Ahora que la computadora juegue **300 partidas** con secretos al azar. Contamos con un **diccionario** cuántas partidas necesitaron 1, 2, 3… intentos (como en la Lección 9) y lo graficamos (como en la Lección 10). Tarda unos segundos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

conteo = {}
todos = []
for _ in range(300):
    secreto = random.choice(candidatos)
    n = computadora_juega(secreto)
    todos.append(n)
    conteo[n] = conteo.get(n, 0) + 1

for n in sorted(conteo):
    print(f"{n} intentos: {conteo[n]} partidas")

print(f"\nPromedio: {np.mean(todos):.2f} intentos   ·   Peor partida: {max(todos)} intentos")

plt.bar(list(conteo.keys()), list(conteo.values()), color="cornflowerblue")
plt.xlabel("intentos para ganar")
plt.ylabel("partidas")
plt.title("300 partidas de la computadora")
plt.show()

La computadora gana casi siempre en 4, 5, 6 o 7 intentos, y lo más común es 5 o 6, con una estrategia que cabe en diez líneas. ✏️ *¿Cómo te fue a ti comparado con ella?*

¿Y la pregunta del principio? Esta estrategia **a veces** necesita 8 o incluso 9 intentos. Los matemáticos demostraron (con computadoras) que con una estrategia más astuta **siempre** se puede ganar en 7 o menos. Encontrarla es el reto de esta lección.

## 9. Errores y trampas comunes

**Trampa 1: el `except` sin tipo.** Se puede escribir `except:` sin decir el tipo de error, y atrapa **todos** los errores… incluso los que son culpa nuestra. Aquí hay un error de escritura (`textoo`), pero el mensaje culpa al jugador:

In [ ]:
texto = "25"
try:
    n = int(textoo)          # ¡error de escritura! Debería ser texto
except:
    print("Eso no es un número")   # mentira: sí era un número

El error real era un `NameError`, y quedó escondido. Con `except ValueError:` Python nos habría mostrado el `NameError` de inmediato. **Regla: atrapa solo el error que esperas.**

**Trampa 2: un `try` gigante.** Si pones veinte líneas dentro de un `try`, no sabrás cuál falló. Pon dentro del `try` solo la línea que puede fallar.

**Trampa 3: olvidar que `input` siempre da texto.** `input()` devuelve un `str`, aunque el jugador escriba un número. Sumar sin convertir da `TypeError`:

In [ ]:
edad = "15"            # así llega un dato desde input()
print(edad + 1)

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …ejecutas `int("4.0")`? ¿Y `float("4.0")`? ¿Y `int(float("4.0"))`?
2. …en la primera celda de la sección 2 cambias `except ValueError:` por `except ZeroDivisionError:`?
3. …llamas `validar_intento("12345", n=5)`? ¿Y `validar_intento("1234", n=5)`?
4. …calculas `contar_picas_fijas("1234", "4321")`? ¿Y `contar_picas_fijas("1234", "1234")`?
5. …cuentas los candidatos si se **permiten** cifras repetidas? ¿Y si son de 3 cifras distintas?
6. …escribes `raise ValueError("¡Hola!")` solo, sin `try`?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (básico). Leer una nota.** Escribe una función `leer_nota()` que pida una nota con `input` y la devuelva como `float`. Debe volver a preguntar si el texto no es un número **o** si la nota no está entre 0 y 5. *Pista: usa `raise ValueError(...)` dentro del `try` cuando la nota esté fuera del rango.*

**Ejercicio 2 (básico). Promedio seguro.** Escribe `promedio(lista)` que devuelva el promedio de una lista de números. Si la lista está vacía, en lugar de caerse con `ZeroDivisionError` debe mostrar un aviso y devolver `None`. Pruébala con `[3.5, 4.0, 4.5]` y con `[]`.

**Ejercicio 3 (intermedio). Modo fácil.** Cambia el juego para jugar con **3 cifras**. Las funciones `generar_secreto`, `validar_intento` y `contar_picas_fijas` ya funcionan con otro tamaño (¡por eso tienen el parámetro `n` y usan `len`!). Escribe `jugar_facil()` y averigua cuántos secretos de 3 cifras distintas hay.

**Ejercicio 4 (intermedio). Marcador.** Usa un diccionario `records = {}` para guardar el **mejor** resultado (menos intentos) de cada jugador. Pide el nombre con `input`, llama a `jugar()` y actualiza el diccionario solo si el jugador ganó y mejoró su marca. Juega varias rondas y muestra el marcador ordenado.

**Ejercicio 5 (avanzado). Tu propio juego: Nim.** Hay 21 palitos. Por turnos, tú y la computadora toman 1, 2 o 3 palitos. **Quien toma el último gana.** Programa el juego con estas reglas:
- valida con `try` / `except` que el jugador escriba 1, 2 o 3 y que no tome más palitos de los que quedan;
- la computadora juega con esta estrategia: si puede, deja un número de palitos **múltiplo de 4**; si no puede, toma 1.

¿Quién gana si los dos juegan perfecto? ¿Importa quién empieza?

## 🏆 Reto opcional: ganar siempre en 7

1. Haz que la computadora juegue con **los 5040 secretos** (no al azar) y calcula el promedio y el peor caso. Tarda un minuto o menos.
2. Prueba otras estrategias para elegir el intento: el último de la lista de posibles, uno al azar, o el que **deje menos posibles en el peor caso** (esta es la idea astuta, pero es lenta: piensa cómo acelerarla).
3. ¿Logras una estrategia que **nunca** necesite más de 7 intentos?

## 🎬 Cierre: ¡terminaste la Sección 1! 🎓

Hoy aprendiste a:
- Leer un mensaje de error y reconocer su tipo.
- Usar `try` / `except` para que tus programas no se caigan, y `except ... as e` para leer el mensaje.
- Lanzar tus propios errores con `raise` y probar funciones con `assert`.
- Construir un programa grande **por partes**: funciones pequeñas, probadas una por una.

Y en el camino usaste **todo** lo de la sección: variables y textos (Lecciones 2–3), condicionales (4), ciclos (5–6), listas (7), funciones (8), diccionarios y tuplas (9), gráficas (10) y numpy (11).

**¿Y ahora?** Ya sabes programar. En la **Sección 2** usarás Python para explorar matemáticas (primos, fractales, π, probabilidad…), y en la **Sección 3** le enseñarás a la computadora a **aprender de los datos**: machine learning y redes neuronales. 🚀